# 07 Model Training

Train baseline and boosted models from the latest available ML dataset. This notebook is aligned to the current pipeline outputs in `data/processed`, `models`, `logs`, and `outputs`.

## Setup

In [ ]:
import os
import sys
import json
import importlib
from pathlib import Path

import pandas as pd
from IPython.display import display, Image

sys.path.append(os.path.abspath('..'))
from src.utils import load_config
from src.quality_gates import QualityGateRunner
from src.schemas import ML_DATASET_COLUMN_ORDER, ML_FEATURE_COLUMNS
import src.model_training as model_training_module

importlib.reload(model_training_module)
ModelTrainer = model_training_module.ModelTrainer

config = load_config('../configs/config.yaml')
paths = config['paths']
processed_dir = Path('..') / paths['processed_data_dir']
models_dir = Path('..') / 'models'
logs_dir = Path('..') / 'logs'
outputs_dir = Path('..') / 'outputs'

primary_dataset_path = processed_dir / paths['ml_dataset_file']
fallback_dataset_path = processed_dir / 'ml_dataset_labeled.parquet'
dataset_path = primary_dataset_path if primary_dataset_path.exists() else fallback_dataset_path

print(f'Using training dataset: {dataset_path}')


## Sync Final Dataset

Optional but recommended: rebuild `ml_dataset.parquet` from the latest labeled dataset so training always uses the newest validated rows.

In [ ]:
SYNC_FINAL_DATASET = True

labeled_dataset_path = processed_dir / 'ml_dataset_labeled.parquet'

if SYNC_FINAL_DATASET and labeled_dataset_path.exists():
    df_labeled_sync = pd.read_parquet(labeled_dataset_path)
    qg = QualityGateRunner()
    report = qg.run_all(df_labeled_sync, feature_cols=ML_FEATURE_COLUMNS)
    df_sync = qg.compute_feature_quality_score(df_labeled_sync, feature_cols=ML_FEATURE_COLUMNS)

    final_cols = []
    for col in ML_DATASET_COLUMN_ORDER:
        if col in df_sync.columns:
            final_cols.append(col)
        else:
            df_sync[col] = pd.NA
            final_cols.append(col)

    df_sync = df_sync[final_cols].copy()
    primary_dataset_path.parent.mkdir(parents=True, exist_ok=True)
    df_sync.to_parquet(primary_dataset_path, index=False)
    dataset_path = primary_dataset_path
    print(f'Synced final dataset from {labeled_dataset_path} -> {primary_dataset_path}')
    print(f"Quality gates passed: {report.get('overall_passed', False)}")
    print(f"Training-eligible rows: {int(df_sync['training_eligible'].fillna(False).sum()) if 'training_eligible' in df_sync.columns else 0:,}")
else:
    print(f'Skipping sync. Using existing dataset at {dataset_path}')


## Training Hardware

Control whether boosted models should try GPU acceleration. The trainer will still fall back to CPU if GPU training is unavailable or unsupported.

In [ ]:
USE_GPU_FOR_TRAINING = config.get('training', {}).get('use_gpu', False)
config.setdefault('training', {})['use_gpu'] = USE_GPU_FOR_TRAINING
print(f'GPU acceleration requested: {USE_GPU_FOR_TRAINING}')


## Load Dataset

In [ ]:
if not dataset_path.exists():
    raise FileNotFoundError(f'No ML dataset found at {dataset_path}. Run notebook 04 and the weather/label/validate pipeline stages first.')

df = pd.read_parquet(dataset_path)
print(f'Dataset shape: {df.shape}')
display(df.head())

if 'label' in df.columns:
    display(df['label'].value_counts(dropna=False).rename_axis('label').to_frame('count'))


## Readiness Check

Training needs at least two populated label classes after any `training_eligible` filtering.

In [ ]:
df_train_scope = df[df['training_eligible'] == True].copy() if 'training_eligible' in df.columns else df.copy()
label_counts = df_train_scope['label'].value_counts(dropna=False) if 'label' in df_train_scope.columns else pd.Series(dtype='int64')
class_count = int((label_counts > 0).sum())
training_ready = class_count >= 2

print(f'Training scope rows: {len(df_train_scope):,}')
print(f'Observed label classes: {class_count}')
display(label_counts.rename_axis('label').to_frame('count'))

if not training_ready:
    print('Training is blocked right now because the dataset is single-class. You can still inspect artifacts and preprocessing outputs below.')


## Train Models

In [ ]:
trainer = None
results = None
X_train = X_test = y_train = y_test = None

if training_ready:
    trainer = ModelTrainer(config, use_gpu=USE_GPU_FOR_TRAINING)
    X_train, X_test, y_train, y_test = trainer.prepare_data(df)
    print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

    results = trainer.train_all(X_train, X_test, y_train, y_test)
    trainer.generate_comparison_report(results, X_test, y_test)

    try:
        trainer.explain(results, X_test, y_test)
    except Exception as exc:
        print(f'SHAP generation did not complete: {exc}')

    summary_rows = []
    for model_name, payload in results.items():
        metrics = payload['metrics']
        weighted_avg = metrics['classification_report'].get('weighted avg', {})
        summary_rows.append({
            'model': model_name,
            'weighted_f1': metrics['weighted_f1'],
            'precision': weighted_avg.get('precision', 0.0),
            'recall': weighted_avg.get('recall', 0.0),
        })
    display(pd.DataFrame(summary_rows).sort_values('weighted_f1', ascending=False))
else:
    print('Skipped model fitting because training readiness check failed.')


## Generated Artifacts

In [ ]:
artifact_candidates = [
    logs_dir / 'model_comparison.json',
    models_dir / 'logistic_regression.pkl',
    models_dir / 'random_forest.pkl',
    models_dir / 'xgboost.pkl',
    models_dir / 'lightgbm.pkl',
    models_dir / 'feature_importance.json',
    outputs_dir / 'roc_curves.png',
    outputs_dir / 'model_comparison.png',
    outputs_dir / 'shap_summary.png',
]
artifact_df = pd.DataFrame({
    'artifact': [str(path) for path in artifact_candidates],
    'exists': [path.exists() for path in artifact_candidates],
})
display(artifact_df)

comparison_path = logs_dir / 'model_comparison.json'
if comparison_path.exists():
    comparison = json.loads(comparison_path.read_text(encoding='utf-8'))
    display(pd.DataFrame(comparison.get('ranking', [])))


## Preview Key Visuals

In [ ]:
for image_name in ['model_comparison.png', 'roc_curves.png', 'shap_summary.png']:
    image_path = outputs_dir / image_name
    if image_path.exists():
        print(image_name)
        display(Image(filename=str(image_path)))
    else:
        print(f'{image_name} not found yet.')
